In [1]:
# ── Cell 1 — imports and connection ───────────────────────────────────────────
import findings_lib as lib

conn = lib.connect()
print("Connected.")

Connected.


In [2]:
# ── Cell 2 — load mart ────────────────────────────────────────────────────────
df, salary_df, n_no_match_excluded = lib.load_mart(conn)
print(f"Loaded {len(df) + n_no_match_excluded} postings ({n_no_match_excluded} excluded as no_match title) → {len(df)} analyzed")

Loaded 1292 postings (67 excluded as no_match title) → 1225 analyzed


In [3]:
# ── Cell 3 — top level snapshot ───────────────────────────────────────────────
snap = lib.top_level_snapshot(df, salary_df)
print("=== TOP LEVEL ===")
print(f"Total postings:       {snap['total_postings']}")
print(f"Unique companies:     {snap['unique_companies']}")
print(f"Role types:           {snap['role_types']}")
print(f"Sources:              {snap['sources']}")
print(f"Salary disclosed:     {snap['salary_disclosed_n']} ({snap['salary_disclosed_rate']:.0%})")
print(f"LLM enriched:         {snap['llm_enriched_n']} ({snap['llm_enriched_rate']:.0%})")
print(f"Date range:           {snap['date_min']} → {snap['date_max']}")
print(f"Last ingested:        {snap['last_ingested']}")

=== TOP LEVEL ===
Total postings:       1225
Unique companies:     901
Role types:           4
Sources:              3
Salary disclosed:     790 (64%)
LLM enriched:         1224 (100%)
Date range:           2026-05-27 → 2026-08-20
Last ingested:        2026-08-20


In [4]:
# ── Cell 4 — postings by role type ───────────────────────────────────────────
print("\n=== POSTINGS BY ROLE TYPE ===")
print(lib.postings_by_role(df).to_string())


=== POSTINGS BY ROLE TYPE ===
title_role_bucket
Data Analyst          460
Data Engineer         351
Data Scientist        324
Analytics Engineer     90


In [5]:
# ── Cell 5 — postings by source ──────────────────────────────────────────────
print("\n=== POSTINGS BY SOURCE ===")
print(lib.postings_by_source(df).to_string())

print("\n=== POSTINGS BY ROLE × SOURCE ===")
print(lib.postings_by_role_source(df).to_string())


=== POSTINGS BY SOURCE ===
source
theirstack    427
builtin       409
jsearch       389

=== POSTINGS BY ROLE × SOURCE ===
source              builtin  jsearch  theirstack
title_role_bucket                               
Analytics Engineer       27       23          40
Data Analyst            163      147         150
Data Engineer           107      121         123
Data Scientist          112       98         114


In [6]:
# ── Cell 6 — work model ───────────────────────────────────────────────────────
wm_overall, wm_by_role = lib.work_model(df)
print("\n=== WORK MODEL (overall) ===")
print(wm_overall.to_string())

print("\n=== WORK MODEL by ROLE ===")
print(wm_by_role.to_string())


=== WORK MODEL (overall) ===
work_model
onsite    739
remote    357
hybrid    129

=== WORK MODEL by ROLE ===
work_model          hybrid  onsite  remote
title_role_bucket                         
Analytics Engineer      16      50      24
Data Analyst            41     284     135
Data Engineer           38     200     113
Data Scientist          34     205      85


In [7]:
# ── Cell 7 — early-career tier distribution ──────────────────────────────────
tier_overall, tier_by_role = lib.early_career_tier(df)
print("\n=== EARLY CAREER TIER (overall) ===")
print(tier_overall.to_string())

print("\n=== EARLY CAREER TIER by ROLE ===")
print(tier_by_role.to_string())


=== EARLY CAREER TIER (overall) ===
early_career_tier
mid                646
entry_or_junior    179

=== EARLY CAREER TIER by ROLE ===
early_career_tier   entry_or_junior  mid
title_role_bucket                       
Analytics Engineer                5   62
Data Analyst                     86  216
Data Engineer                    39  191
Data Scientist                   49  177


In [8]:
# ── Cell 8 — salary by role ───────────────────────────────────────────────────
sal_by_role, sal_by_role_tier = lib.salary_by_role(df, salary_df)
print("\n=== SALARY by ROLE (median, where disclosed) ===")
print(sal_by_role.to_string())

print("\n=== SALARY by ROLE × EARLY CAREER TIER ===")
print(sal_by_role_tier.to_string())


=== SALARY by ROLE (median, where disclosed) ===
                      median    n      min        max      std
title_role_bucket                                             
Analytics Engineer  133312.0   66  80000.0   445000.0  52349.0
Data Analyst         93502.0  274   3000.0   210000.0  30334.0
Data Engineer       130000.0  210  42000.0  1100000.0  89826.0
Data Scientist      142064.0  240      2.0   450000.0  64275.0

=== SALARY by ROLE × EARLY CAREER TIER ===
                                        median  count
title_role_bucket  early_career_tier                 
Analytics Engineer entry_or_junior     95000.0      5
                   mid                132500.0     47
Data Analyst       entry_or_junior     90000.0     49
                   mid                100500.0    132
Data Engineer      entry_or_junior    100000.0     23
                   mid                135000.0    117
Data Scientist     entry_or_junior    103750.0     38
                   mid                1460

In [9]:
# ── Cell 9 — AI blindspot ─────────────────────────────────────────────────────
ai_overall, ai_by_role = lib.ai_acknowledgment(df)
print("\n=== AI ACKNOWLEDGMENT (overall) ===")
print(f"Acknowledges AI: {ai_overall['yes']} of {ai_overall['total']} ({ai_overall['rate']:.0%})")

print("\n=== AI ACKNOWLEDGMENT by ROLE ===")
print(ai_by_role.to_string())


=== AI ACKNOWLEDGMENT (overall) ===
Acknowledges AI: 652 of 1225 (53%)

=== AI ACKNOWLEDGMENT by ROLE ===
                    yes  total   rate
title_role_bucket                    
Analytics Engineer   55     90  0.611
Data Analyst        148    460  0.322
Data Engineer       170    351  0.484
Data Scientist      279    324  0.861


In [10]:
# ── Cell 10 — title vs archetype confusion ────────────────────────────────────
consistency = lib.title_vs_archetype(df)
print("\n=== TITLE vs LLM ARCHETYPE (confusion matrix, counts) ===")
print(consistency["pivot_counts"].to_string())

print("\n=== TITLE vs LLM ARCHETYPE (row %, agreement on diagonal) ===")
print(consistency["pivot_pct"].to_string())

print(f"\nAgreement: {consistency['overall_agree_n']} of {consistency['overall_agree_total']} ({consistency['overall_agree_rate']:.0%})")
print("\nAgreement by role:")
print(consistency["by_role_rate"].round(3).to_string())


=== TITLE vs LLM ARCHETYPE (confusion matrix, counts) ===
role_archetype      analytics_engineer  data_analyst  data_engineer  data_scientist  hybrid
title_role_bucket                                                                          
Analytics Engineer                  83             0              3               0       4
Data Analyst                         2           442              3               3      10
Data Engineer                        1             2            341               0       6
Data Scientist                       0             0              3             317       4

=== TITLE vs LLM ARCHETYPE (row %, agreement on diagonal) ===
role_archetype      analytics_engineer  data_analyst  data_engineer  data_scientist  hybrid
title_role_bucket                                                                          
Analytics Engineer                0.92          0.00           0.03            0.00    0.04
Data Analyst                      0.00          0.

In [11]:
# ── Cell 11 — top skills overall and by role ──────────────────────────────────
top_overall, top_by_role = lib.top_skills(df)
print("\n=== TOP 20 REQUIRED SKILLS (overall) ===")
print(top_overall.to_string())

print("\n=== TOP 10 REQUIRED SKILLS by ROLE ===")
for role, series in top_by_role.items():
    print(f"\n--- {role} ---")
    print(series.to_string())


=== TOP 20 REQUIRED SKILLS (overall) ===
sql           834
python        690
excel         203
snowflake     179
r             145
tableau       144
power bi      135
dbt           128
aws           119
databricks     91
airflow        85
bigquery       79
spark          72
git            64
pyspark        60
looker         55
redshift       54
azure          52
pandas         43
scala          42

=== TOP 10 REQUIRED SKILLS by ROLE ===

--- Data Analyst ---
sql          280
excel        184
python       120
tableau       95
power bi      92
r             44
looker        37
snowflake     32
sas           18
powerbi       17

--- Data Engineer ---
python        255
sql           251
snowflake      90
aws            75
databricks     61
airflow        59
dbt            57
spark          52
pyspark        38
bigquery       37

--- Data Scientist ---
python          274
sql             232
r                90
pandas           34
scikit-learn     28
numpy            27
tableau          26

In [12]:
# ── Cell 12 — top paradigms by role ──────────────────────────────────────────
paradigms_by_role = lib.top_paradigms(df)
print("\n=== TOP 10 PARADIGMS by ROLE ===")
for role, series in paradigms_by_role.items():
    print(f"\n--- {role} ---")
    print(series.to_string())


=== TOP 10 PARADIGMS by ROLE ===

--- Data Analyst ---
data analysis            171
data quality             159
data visualization       131
data governance          102
data modeling             70
statistical analysis      67
data validation           64
data management           38
reporting                 30
business intelligence     27

--- Data Engineer ---
etl design                206
data quality              179
data modeling             167
data governance           133
data warehousing           95
ci/cd                      44
data integration           43
pipeline orchestration     28
agile                      20
data lineage               17

--- Data Scientist ---
machine learning        128
statistical analysis    116
data analysis            61
data modeling            48
data quality             48
causal inference         45
predictive modeling      40
data visualization       38
statistical modeling     35
experimental design      33

--- Analytics Engineer ---

In [13]:
# ── Cell 13 — experience requirements ────────────────────────────────────────
yrs_by_role, yrs_by_role_tier = lib.years_required(df)
print("\n=== YEARS REQUIRED by ROLE (median, where specified) ===")
print(yrs_by_role.to_string())

print("\n=== YEARS REQUIRED by ROLE × EARLY CAREER TIER ===")
print(yrs_by_role_tier.to_string())


=== YEARS REQUIRED by ROLE (median, where specified) ===
                    median  count
title_role_bucket                
Analytics Engineer     3.0     85
Data Analyst           2.0    407
Data Engineer          3.0    321
Data Scientist         3.0    303

=== YEARS REQUIRED by ROLE × EARLY CAREER TIER ===
                                      median  count
title_role_bucket  early_career_tier               
Analytics Engineer entry_or_junior       2.0      5
                   mid                   3.0     58
Data Analyst       entry_or_junior       2.0     69
                   mid                   3.0    199
Data Engineer      entry_or_junior       1.0     31
                   mid                   3.0    178
Data Scientist     entry_or_junior       1.0     46
                   mid                   3.0    167


In [14]:
# ── Cell 14 — degree requirements ────────────────────────────────────────────
deg_counts, deg_pct = lib.degree_requirements(df)
print("\n=== DEGREE REQUIREMENTS by ROLE ===")
print(deg_counts.to_string())


=== DEGREE REQUIREMENTS by ROLE ===
degree_requirement  bachelors  equivalent_ok  masters  none
title_role_bucket                                          
Analytics Engineer         39              3        0    48
Data Analyst              250             49       15   146
Data Engineer             141             18       12   179
Data Scientist            129             21       83    91


In [15]:
# ── Cell 15 — encourages applicants ──────────────────────────────────────────
enc_overall, enc_by_role = lib.encourages_applicants(df)
print("\n=== ENCOURAGES APPLICANTS (overall) ===")
print(f"Yes: {enc_overall['yes']} of {enc_overall['total']} ({enc_overall['rate']:.1%})")

print("\n=== ENCOURAGES APPLICANTS by ROLE ===")
print(enc_by_role.to_string())


=== ENCOURAGES APPLICANTS (overall) ===
Yes: 148 of 1225 (12.1%)

=== ENCOURAGES APPLICANTS by ROLE ===
                    yes  total   rate
title_role_bucket                    
Analytics Engineer   15     90  0.167
Data Analyst         44    460  0.096
Data Engineer        34    351  0.097
Data Scientist       55    324  0.170


In [16]:
# ── Cell 16 — listed vs. inferred seniority mismatch ──────────────────────────
seniority = lib.seniority_mismatch(df)
print("\n=== LISTED vs INFERRED SENIORITY (counts) ===")
print(seniority["pivot_counts"].to_string())

print("\n=== LISTED vs INFERRED SENIORITY (row %, agreement on diagonal) ===")
print(seniority["pivot_pct"].to_string())

print(f"\nTotal rows with both fields populated: {seniority['n_both_populated']}")

print("\n=== YEARS REQUIRED by LISTED SENIORITY (overlap check) ===")
print(seniority["years_by_listed_seniority"].to_string())


=== LISTED vs INFERRED SENIORITY (counts) ===
inferred_seniority  entry  junior  mid  senior
listed_seniority                              
entry_level            21       3    0       0
junior                 29     119    5       1
mid_level              49     150  353      94

=== LISTED vs INFERRED SENIORITY (row %, agreement on diagonal) ===
inferred_seniority  entry  junior   mid  senior
listed_seniority                               
entry_level          0.88    0.12  0.00    0.00
junior               0.19    0.77  0.03    0.01
mid_level            0.08    0.23  0.55    0.15

Total rows with both fields populated: 824

=== YEARS REQUIRED by LISTED SENIORITY (overlap check) ===
                  median  min   max  count
listed_seniority                          
entry_level          0.0  0.0   2.0     10
junior               2.0  0.0   5.0    141
mid_level            3.0  0.0  12.0    602
